In [1]:
# Cell 1: Imports & Environment Setup
import os
import pandas as pd
import numpy as np
import torch
import torchaudio
import torchaudio.transforms as T
import matplotlib.pyplot as plt
from pathlib import Path
from torch.utils.data import Dataset, DataLoader

# Fix random seeds for reproducibility
torch.manual_seed(42)
np.random.seed(42)

# Check for GPU (RTX 3060 12GB)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")
if torch.cuda.is_available():
    print(f"GPU Name: {torch.cuda.get_device_name(0)}")
    print(f"VRAM Available: {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB")

# Define base paths exactly as specified in the project structure
BASE_DIR = Path.cwd().parent  # Adjust this depending on where your notebook is started from
AUDIO_DIR = BASE_DIR / "Data" / "MELD.Audio"
RAW_DIR = BASE_DIR / "Data" / "MELD.Raw"

# Define specific splits
AUDIO_TRAIN = AUDIO_DIR / "train"
AUDIO_DEV = AUDIO_DIR / "dev"
AUDIO_TEST = AUDIO_DIR / "test"

# Metadata CSV paths (we need these for the emotion labels)
META_TRAIN = RAW_DIR / "train" / "train_sent_emo.csv"
META_DEV = RAW_DIR / "dev_sent_emo.csv"
META_TEST = RAW_DIR / "test_sent_emo.csv"

# Emotion mapping (MELD classes)
EMOTION_TO_IDX = {
    'neutral': 0,
    'surprise': 1,
    'fear': 2,
    'sadness': 3, # Usually 'sadness' or 'sad' in MELD, we'll verify when loading
    'joy': 4,     # Usually 'joy' or 'happy'
    'disgust': 5,
    'anger': 6
}

print("\nPaths configured successfully!")
print(f"Audio Train Path: {AUDIO_TRAIN}")
print(f"Metadata Train Path: {META_TRAIN}")

Using device: cuda
GPU Name: NVIDIA GeForce RTX 3060
VRAM Available: 12.88 GB

Paths configured successfully!
Audio Train Path: e:\Licenta\Multimodal-Emotion-Recognition---Bachelor-Thesis\Data\MELD.Audio\train
Metadata Train Path: e:\Licenta\Multimodal-Emotion-Recognition---Bachelor-Thesis\Data\MELD.Raw\train\train_sent_emo.csv


In [2]:
# Cell 2: Robust MELD Audio Dataset
class MeldAudioDataset(Dataset):
    def __init__(self, csv_file, audio_dir, emotion_map, target_sr=16000):
        self.audio_dir = Path(audio_dir)
        self.emotion_map = emotion_map
        self.target_sr = target_sr
        
        try:
            self.df = pd.read_csv(csv_file)
            # Curățăm numele coloanelor de spații invizibile și le facem litere mici
            self.df.columns = self.df.columns.str.strip().str.lower()
        except Exception as e:
            print(f"Error la citirea CSV: {e}")
            return

        self.samples = []
        
        # Detectăm numele coloanelor indiferent de format
        # Căutăm coloana care conține 'dialogue' și cea care conține 'utterance'
        col_dia = [c for c in self.df.columns if 'dialogue' in c][0]
        col_utt = [c for c in self.df.columns if 'utterance' in c and 'id' in c][0]
        col_emo = [c for c in self.df.columns if 'emotion' in c][0]
        
        print(f"Coloane detectate: Dia={col_dia}, Utt={col_utt}, Emo={col_emo}")

        missing_count = 0
        for _, row in self.df.iterrows():
            dia_id = int(row[col_dia])
            utt_id = int(row[col_utt])
            emotion = str(row[col_emo]).strip().lower()
            
            # Mapări de rezervă pentru etichete
            if emotion == 'happy': emotion = 'joy'
            if emotion == 'sad': emotion = 'sadness'
            
            if emotion not in self.emotion_map:
                continue
                
            filename = f"dia{dia_id}_utt{utt_id}.wav"
            file_path = self.audio_dir / filename
            
            if file_path.exists():
                self.samples.append({
                    'path': str(file_path),
                    'label': self.emotion_map[emotion]
                })
            else:
                missing_count += 1
                
        print(f"Dataset finalizat: {len(self.samples)} fișiere. Lipsă pe disk: {missing_count}")

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        waveform, sr = torchaudio.load(self.samples[idx]['path'])
        if waveform.shape[0] > 1:
            waveform = torch.mean(waveform, dim=0, keepdim=True)
        if sr != self.target_sr:
            waveform = T.Resample(sr, self.target_sr)(waveform)
        return waveform, self.samples[idx]['label']

# Re-testăm
train_dataset = MeldAudioDataset(META_TRAIN, AUDIO_TRAIN, EMOTION_TO_IDX)

Coloane detectate: Dia=dialogue_id, Utt=utterance_id, Emo=emotion
Dataset finalizat: 9988 fișiere. Lipsă pe disk: 1


In [16]:
# Cell 3: Dynamic Padding Collate Function & DataLoaders
from torch.nn.utils.rnn import pad_sequence

# Parametri pentru DataLoaders
BATCH_SIZE = 16 # O valoare sigură pentru început pe 12GB VRAM
MAX_DURATION_SECONDS = 8 # Tăiem orice depășește 8 secunde
SAMPLE_RATE = 16000
MAX_SAMPLES = MAX_DURATION_SECONDS * SAMPLE_RATE

def dynamic_collate_fn(batch):
    """
    Funcție pentru a face padding dinamic batch-urilor.
    Returnează:
    - waveforms: (Batch, 1, Length)
    - labels: (Batch)
    - lengths: (Batch) - util pentru Attention Masks în Wav2Vec2
    """
    waveforms = []
    labels = []
    lengths = []
    
    for waveform, label in batch:
        # Eliminăm temporar dimensiunea canalului (1, L) -> (L) pentru pad_sequence
        wave = waveform.squeeze(0) 
        
        # Aplicăm "plasa de siguranță" (Truncate) dacă este prea lung
        if wave.shape[0] > MAX_SAMPLES:
            wave = wave[:MAX_SAMPLES]
            
        waveforms.append(wave)
        labels.append(label)
        lengths.append(len(wave))
        
    # Padding dinamic la cel mai lung element DIN ACEST BATCH
    padded_waveforms = pad_sequence(waveforms, batch_first=True)
    
    # Readăugăm dimensiunea canalului: (Batch, L) -> (Batch, 1, L)
    padded_waveforms = padded_waveforms.unsqueeze(1)
    
    # Convertim listele în tensori PyTorch
    labels = torch.tensor(labels, dtype=torch.long)
    lengths = torch.tensor(lengths, dtype=torch.long)
    
    return padded_waveforms, labels, lengths

# Inițializăm restul dataset-urilor (presupunem că train_dataset e deja creat)
print("Inițializare Dev & Test Datasets...")
dev_dataset = MeldAudioDataset(META_DEV, AUDIO_DEV, EMOTION_TO_IDX)
test_dataset = MeldAudioDataset(META_TEST, AUDIO_TEST, EMOTION_TO_IDX)

# Creăm DataLoaders
train_loader = DataLoader(
    train_dataset, 
    batch_size=BATCH_SIZE, 
    shuffle=True, 
    collate_fn=dynamic_collate_fn,
    num_workers=0, # Poți ajusta în funcție de procesorul tău (CPU)
    pin_memory=True if torch.cuda.is_available() else False
)

dev_loader = DataLoader(
    dev_dataset, 
    batch_size=BATCH_SIZE, 
    shuffle=False, 
    collate_fn=dynamic_collate_fn,
    num_workers=0,
    pin_memory=True if torch.cuda.is_available() else False
)

test_loader = DataLoader(
    test_dataset, 
    batch_size=BATCH_SIZE, 
    shuffle=False, 
    collate_fn=dynamic_collate_fn,
    num_workers=0,
    pin_memory=True if torch.cuda.is_available() else False
)

# Testăm ieșirea primului batch pentru a valida dimensiunile
for batch_waveforms, batch_labels, batch_lengths in train_loader:
    print("\nValidare Batch Train:")
    print(f"Shape Waveforms: {batch_waveforms.shape} -> (Batch, Channel, Length)")
    print(f"Shape Labels: {batch_labels.shape}")
    print(f"Lengths din batch: {batch_lengths.tolist()}")
    break # Ieșim după primul batch

Inițializare Dev & Test Datasets...
Coloane detectate: Dia=dialogue_id, Utt=utterance_id, Emo=emotion
Dataset finalizat: 1108 fișiere. Lipsă pe disk: 1
Coloane detectate: Dia=dialogue_id, Utt=utterance_id, Emo=emotion
Dataset finalizat: 2610 fișiere. Lipsă pe disk: 0

Validare Batch Train:
Shape Waveforms: torch.Size([16, 1, 91477]) -> (Batch, Channel, Length)
Shape Labels: torch.Size([16])
Lengths din batch: [31403, 91477, 90112, 69632, 56320, 32085, 42325, 11264, 41301, 46763, 49493, 62805, 29355, 32085, 37547, 31403]


In [4]:
# Cell 4: Training and Validation Utilities
from tqdm import tqdm
import torch.nn as nn

def train_one_epoch(model, loader, optimizer, criterion, device):
    model.train()
    running_loss = 0.0
    correct = 0
    total = 0
    
    # Folosim tqdm pentru o bară de progres vizuală în Jupyter
    pbar = tqdm(loader, desc="Training")
    for waveforms, labels, lengths in pbar:
        waveforms, labels = waveforms.to(device), labels.to(device)
        
        optimizer.zero_grad()
        
        # Forward pass
        # Notă: Unele modele (ca Wav2Vec2) pot folosi 'lengths', dar ResNet nu.
        # Încercăm să pasăm doar waveforms pentru simplitate acum.
        outputs = model(waveforms)
        
        loss = criterion(outputs, labels)
        
        # Backward pass
        loss.backward()
        optimizer.step()
        
        # Statistici
        running_loss += loss.item()
        _, predicted = outputs.max(1)
        total += labels.size(0)
        correct += predicted.eq(labels).sum().item()
        
        pbar.set_postfix({'loss': f"{loss.item():.4f}", 'acc': f"{100.*correct/total:.2f}%"})
        
    return running_loss / len(loader), 100. * correct / total

def validate(model, loader, criterion, device):
    model.eval()
    running_loss = 0.0
    correct = 0
    total = 0
    
    with torch.no_grad():
        for waveforms, labels, _ in loader:
            waveforms, labels = waveforms.to(device), labels.to(device)
            outputs = model(waveforms)
            loss = criterion(outputs, labels)
            
            running_loss += loss.item()
            _, predicted = outputs.max(1)
            total += labels.size(0)
            correct += predicted.eq(labels).sum().item()
            
    return running_loss / len(loader), 100. * correct / total

print("Funcțiile de train/val au fost definite.")

Funcțiile de train/val au fost definite.


In [5]:
# Cell 5: Light Model Definition (ResNet18 for Mel-Spectrograms)
import torch.nn as nn
from torchvision import models
import torchaudio.transforms as T

class AudioLightModel(nn.Module):
    def __init__(self, num_classes=7, sample_rate=16000):
        super(AudioLightModel, self).__init__()
        
        # Transformare Audio -> Imagine (Spectrogramă)
        self.mel_spec = T.MelSpectrogram(
            sample_rate=sample_rate,
            n_fft=1024,
            hop_length=512,
            n_mels=64
        )
        self.amplitude_to_db = T.AmplitudeToDB()
        
        # ResNet18 preantrenat
        self.resnet = models.resnet18(weights=models.ResNet18_Weights.DEFAULT)
        
        # Ajustăm primul strat: de la 3 canale (RGB) la 1 canal (Grayscale/Audio)
        self.resnet.conv1 = nn.Conv2d(1, 64, kernel_size=7, stride=2, padding=3, bias=False)
        
        # Ajustăm ultimul strat pentru cele 7 emoții MELD
        self.resnet.fc = nn.Linear(self.resnet.fc.in_features, num_classes)

    def forward(self, x):
        # x: (Batch, 1, Time)
        x = self.mel_spec(x)       # -> (Batch, 1, Mels, Time)
        x = self.amplitude_to_db(x)
        return self.resnet(x)

# Instanțiere
light_model = AudioLightModel(num_classes=len(EMOTION_TO_IDX)).to(device)

Downloading: "https://download.pytorch.org/models/resnet18-f37072fd.pth" to E:\YOLO_temp_data\torch\hub\checkpoints\resnet18-f37072fd.pth


100%|██████████| 44.7M/44.7M [00:00<00:00, 114MB/s]


In [7]:
from sklearn.metrics import f1_score, classification_report

def validate(model, loader, criterion, device):
    model.eval()
    running_loss = 0.0
    all_preds = []
    all_labels = []
    
    with torch.no_grad():
        for waveforms, labels, _ in loader:
            waveforms, labels = waveforms.to(device), labels.to(device)
            outputs = model(waveforms)
            loss = criterion(outputs, labels)
            
            running_loss += loss.item()
            _, predicted = outputs.max(1)
            
            all_preds.extend(predicted.cpu().numpy())
            all_labels.extend(labels.cpu().numpy())
            
    # Calculăm F1-Score
    # 'weighted' ține cont de dezechilibrul claselor (bun pentru performanța generală)
    # 'macro' dă importanță egală fiecărei clase (bun pentru a vedea dacă înveți clasele rare)
    f1_w = f1_score(all_labels, all_preds, average='weighted')
    f1_m = f1_score(all_labels, all_preds, average='macro')
    
    # Putem printa și un raport complet pentru a vedea precision/recall per emoție
    # report = classification_report(all_labels, all_preds, zero_division=0)
    
    return running_loss / len(loader), f1_w, f1_m

In [11]:
# Cell 6: Run Training with Best Model Saving (F1-Score based)
import torch.optim as optim
import copy

# Hyperparametri
EPOCHS = 15
LEARNING_RATE = 1e-4
best_val_f1 = 0.0
model_path = "best_audio_light_model.pth"

# Re-inițializăm modelul și optimizer-ul
light_model = AudioLightModel(num_classes=len(EMOTION_TO_IDX)).to(device)
optimizer = optim.AdamW(light_model.parameters(), lr=LEARNING_RATE, weight_decay=0.01)
criterion = nn.CrossEntropyLoss() # Poți folosi și varianta cu weights calculată anterior

history = {"train_loss": [], "train_acc": [], "val_f1_w": [], "val_f1_m": []}

print(f"Începe antrenarea... Se va salva cel mai bun model în: {model_path}")

for epoch in range(EPOCHS):
    # 1. Train
    t_loss, t_acc = train_one_epoch(light_model, train_loader, optimizer, criterion, device)
    
    # 2. Validation (folosind noua funcție cu F1)
    v_loss, v_f1_w, v_f1_m = validate(light_model, dev_loader, criterion, device)
    
    # Salvează metricile
    history["train_loss"].append(t_loss)
    history["train_acc"].append(t_acc)
    history["val_f1_w"].append(v_f1_w)
    history["val_f1_m"].append(v_f1_m)
    
    # 3. Checkpoint Logic: Salvăm dacă F1-Weighted s-a îmbunătățit
    if v_f1_w > best_val_f1:
        best_val_f1 = v_f1_w
        # Salvăm starea modelului pe disk
        torch.save(light_model.state_dict(), model_path)
        print(f"--> Epoch {epoch+1}: New Best F1-W: {v_f1_w:.4f}! Model salvat.")
    
    print(f"Epoch {epoch+1}/{EPOCHS} | Loss: {t_loss:.4f} | Val F1-W: {v_f1_w:.4f} | Val F1-M: {v_f1_m:.4f}")

print(f"\nAntrenare gata! Cel mai bun F1-Weighted atins: {best_val_f1:.4f}")

# Opțional: Încarcă cel mai bun model înapoi în memorie pentru testare
light_model.load_state_dict(torch.load(model_path))
print("Modelul optim a fost reîncărcat pentru evaluarea finală.")

Începe antrenarea... Se va salva cel mai bun model în: best_audio_light_model.pth


Training: 100%|██████████| 625/625 [00:15<00:00, 39.61it/s, loss=1.6390, acc=45.67%]


--> Epoch 1: New Best F1-W: 0.2890! Model salvat.
Epoch 1/15 | Loss: 1.5537 | Val F1-W: 0.2890 | Val F1-M: 0.1205


Training: 100%|██████████| 625/625 [00:15<00:00, 40.38it/s, loss=0.8127, acc=46.93%]


--> Epoch 2: New Best F1-W: 0.3276! Model salvat.
Epoch 2/15 | Loss: 1.4997 | Val F1-W: 0.3276 | Val F1-M: 0.1592


...

Epoch 3/15 | Loss: 1.4817 | Val F1-W: 0.3124 | Val F1-M: 0.1514


Training: 100%|██████████| 625/625 [00:15<00:00, 40.27it/s, loss=1.8740, acc=48.09%]


Epoch 4/15 | Loss: 1.4585 | Val F1-W: 0.3095 | Val F1-M: 0.1411


Training:  37%|███▋      | 230/625 [00:05<00:10, 39.43it/s, loss=1.3208, acc=49.35%]


KeyboardInterrupt: 

In [9]:
val_loss, val_f1_weighted, val_f1_macro = validate(light_model, dev_loader, criterion, device)

print(f"Validation Results - Loss: {val_loss:.4f} | F1 Weighted: {val_f1_weighted:.4f} | F1 Macro: {val_f1_macro:.4f}")

Validation Results - Loss: 2.7688 | F1 Weighted: 0.3110 | F1 Macro: 0.1775


In [12]:
# Cell 7: Heavy Model Definition (Wav2Vec2 for Sequence Classification)
from transformers import Wav2Vec2ForSequenceClassification

class AudioHeavyModel(nn.Module):
    def __init__(self, num_classes=7):
        super(AudioHeavyModel, self).__init__()
        
        # Încărcăm modelul preantrenat pe vorbire (960h de audio din Librispeech)
        self.model = Wav2Vec2ForSequenceClassification.from_pretrained(
            "facebook/wav2vec2-base-960h", 
            num_labels=num_classes
        )
        
        # Înghețăm extractorul de trăsături (straturile CNN de bază) 
        # pentru a face antrenarea mai stabilă și a economisi VRAM
        self.model.freeze_feature_encoder()

    def forward(self, x):
        # x are forma (Batch, 1, Length) -> Wav2Vec2 vrea (Batch, Length)
        x = x.squeeze(1)
        
        # Returnăm direct logit-ii
        output = self.model(x)
        return output.logits

# Instanțiere
heavy_model = AudioHeavyModel(num_classes=len(EMOTION_TO_IDX)).to(device)
print(f"Model Heavy încărcat. Parametri: {sum(p.numel() for p in heavy_model.parameters()):,}")

e:\Licenta\Multimodal-Emotion-Recognition---Bachelor-Thesis\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
e:\Licenta\Multimodal-Emotion-Recognition---Bachelor-Thesis\.venv\Lib\site-packages\huggingface_hub\file_download.py:130: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in D:\YOLO_Cache\huggingface\hub\models--facebook--wav2vec2-base-960h. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to

Model Heavy încărcat. Parametri: 94,570,375


In [18]:
# Cell 8: Training for Heavy Model
import torch.optim as optim

# Parametri adaptați pentru Transformer
EPOCHS_HEAVY = 100
LR_HEAVY = 2e-5 
best_heavy_f1 = 0.0
heavy_model_path = "best_audio_heavy_model.pth"

# Optimizer și Criterion
optimizer_heavy = optim.AdamW(heavy_model.parameters(), lr=LR_HEAVY)
criterion = nn.CrossEntropyLoss() # Sau varianta ta cu weights

history_heavy = {"train_loss": [], "train_acc": [], "val_f1_w": [], "val_f1_m": []}

print(f"Începe antrenarea Heavy Model pe {device}...")

for epoch in range(EPOCHS_HEAVY):
    # 1. Train
    t_loss, t_acc = train_one_epoch(heavy_model, train_loader, optimizer_heavy, criterion, device)
    
    # 2. Validation
    v_loss, v_f1_w, v_f1_m = validate(heavy_model, dev_loader, criterion, device)
    
    # Salvează istoricul
    history_heavy["train_loss"].append(t_loss)
    history_heavy["train_acc"].append(t_acc)
    history_heavy["val_f1_w"].append(v_f1_w)
    history_heavy["val_f1_m"].append(v_f1_m)
    
    # 3. Checkpoint pe F1-Weighted
    if v_f1_w > best_heavy_f1:
        best_heavy_f1 = v_f1_w
        torch.save(heavy_model.state_dict(), heavy_model_path)
        print(f"--> Heavy New Best F1-W: {v_f1_w:.4f}! Model salvat.")
    
    print(f"Epoch {epoch+1} | Loss: {t_loss:.4f} | Val F1-W: {v_f1_w:.4f} | Val F1-M: {v_f1_m:.4f}")

print(f"\nAntrenare Heavy gata! Cel mai bun F1-W: {best_heavy_f1:.4f}")

Începe antrenarea Heavy Model pe cuda...


Training: 100%|██████████| 625/625 [07:04<00:00,  1.47it/s, loss=1.3844, acc=47.97%]


--> Heavy New Best F1-W: 0.3633! Model salvat.
Epoch 1 | Loss: 1.4678 | Val F1-W: 0.3633 | Val F1-M: 0.1837


Training: 100%|██████████| 625/625 [07:05<00:00,  1.47it/s, loss=1.2687, acc=49.06%]


Epoch 2 | Loss: 1.4370 | Val F1-W: 0.3629 | Val F1-M: 0.1829


...

Epoch 25 | Loss: 0.5677 | Val F1-W: 0.3945 | Val F1-M: 0.2695


Training: 100%|██████████| 625/625 [12:55<00:00,  1.24s/it, loss=0.3414, acc=82.00%]


Epoch 26 | Loss: 0.5372 | Val F1-W: 0.3898 | Val F1-M: 0.2449


Training:  11%|█         | 70/625 [01:23<11:01,  1.19s/it, loss=0.7378, acc=82.32%]


KeyboardInterrupt: 